# Imports

In [ ]:
import numpy as np
import sys
import os
import contextlib
import cvxpy as cp
import time
import matplotlib.pyplot as plt
import pygad
import pyswarms as ps
import casadi as ca

from tqdm import tqdm
from manim import *
import pandas as pd

# Parameters and Key Values

In [ ]:
mu = 3.986e14 # Gravitational parameter for Earth in m^3/s^2

a = 7000e3 # Semi-major axis in meters

e = 0.1 # Orbital eccentricity

x0 = np.array([0,0,0,0,0,0,0,0,0,1,0,0,0]) # Initial state vector

xf = np.array([5,5,5,0,0,0,0.5,0.5,0.5,0.5,0,0,0]) # Final state vector

nu = np.pi / 4 # Maximum Allowable Thrust Angle

I = 1000*np.diag([3,2,3]) # Intertia Tensor

m = 2000 # Mass in kg

N = 50 # Number of discretization steps

tf = 50 # Final time in seconds

#rs = [np.array([-1, -1, -1]),np.array([0, 1, 0]),np.array([0, -1, 0])]

rs = [np.array([-1, -1, -1]), np.array([0, 1, 0]), np.array([0, -1, 0])]

#rs = rsfull

a0 = x0[6:13]
af = xf[6:13]

epsilon = 1e-6  # Small constant to avoid division by zero

In [ ]:
#Currently running Scenario 2

mu = 3.986e14 # Gravitational parameter for Earth in m^3/s^2

a = 8000e3 # Semi-major axis in meters

e = 0.2 # Orbital eccentricity

x0 = np.array([1,10,-5,0,0.5,0,0.707,0,0.707,0,0,0.1,0]) # Initial state vector

xf = np.array([0,0,0,0,0,0,0,0,0,1,0,0,0]) # Final state vector

nu = np.pi / 4 # Maximum Allowable Thrust Angle

I = 1000*np.diag([1,2,3]) # Intertia Tensor

m = 2000 # Mass in kg

N = 40 # Number of discretization steps

tf = 60 # Final time in seconds

rs = [np.array([-1, -1, -1]),np.array([0, 1, 0]),np.array([0, -1, 0])]

rs = [np.array([0.5, 1, 1.5]), np.array([0, 1.5, 2]), np.array([-0.5, 1, -1.5])]


a0 = x0[6:13]
af = xf[6:13]

epsilon = 1e-12  # Small constant to avoid division by zero

In [ ]:
# #Currently running Scenario 3

# mu = 3.986e14 # Gravitational parameter for Earth in m^3/s^2

# a = 1.05e7 # Semi-major axis in meters

# e = 0.01 # Orbital eccentricity

# x0 = np.array([-50,50,-50,0,0,0,0,0,0,1,-0.01,0.01,0.01]) # Initial state vector

# xf = np.array([5,-5,5,0,0,0,0.5,-0.5,0.5,-0.5,0,0,0]) # Final state vector

# nu = np.pi / 4 # Maximum Allowable Thrust Angle

# I = 1000*np.diag([1,2,3]) # Intertia Tensor

# m = 2000 # Mass in kg

# N = 400 # Number of discretization steps

# tf = 600 # Final time in seconds

# #rs = [np.array([-1, -1, -1]),np.array([0, 1, 0]),np.array([0, -1, 0])]

# rs = [np.array([-1, -1.5, 0.5]), np.array([0.5, -1, -2.5]), np.array([-1, -1, 1.5]), np.array([1, 2.5, 0])]


# a0 = x0[6:13]
# af = xf[6:13]

# epsilon = 1e-12  # Small constant to avoid division by zero

# Helper Functions

## Orbital Functions

In [ ]:
def kepler_eq_newton(M, e, tol=1e-10, max_iter=100):
    E = M  # Initial guess
    for _ in range(max_iter):
        E_next = E - (E - e * np.sin(E) - M) / (1 - e * np.cos(E))
        if abs(E_next - E) < tol:
            return E_next
        E = E_next
    raise RuntimeError("Kepler's equation did not converge")

def true_anomaly(E, e):
    return 2 * np.arctan2(np.sqrt(1 + e) * np.sin(E / 2), np.sqrt(1 - e) * np.cos(E / 2))

def f_dot(n, e, f):
    return n * (1 + e * np.cos(f))**2 / (1 - e**2)**(3/2)

def f_double_dot(n, e, f):
    return -2 * n**2 * e * np.sin(f) * (1 + e * np.cos(f))**3 / (1 - e**2)**3

def calculate_orbital_params(mu, a, e, t, t0=0, f0=0):
    # Calculate the mean motion
    n = np.sqrt(mu / a**3)

    # Calculate the mean anomaly at the current time
    M = n * (t - t0)

    # Solve Kepler's equation for the eccentric anomaly E
    E = kepler_eq_newton(M, e)

    # Calculate the true anomaly f
    f = true_anomaly(E, e)

    # Calculate f_dot
    fdot = f_dot(n, e, f)

    # Calculate f_double_dot
    fddot = f_double_dot(n, e, f)

    return f, fdot, fddot

## Matrix and Vector Functions

In [ ]:
def skew(v):
    """
    Returns the skew-symmetric matrix of a vector v.

    Parameters:
    v (array-like): A 3-element vector.

    Returns:
    np.ndarray: A 3x3 skew-symmetric matrix.
    """
    v = np.asarray(v)
    return np.array([
        [0, -v[2], v[1]],
        [v[2], 0, -v[0]],
        [-v[1], v[0], 0]
    ])

def smooth_norm(vec, delta):
    return ca.sqrt(ca.dot(vec, vec) + delta**2)

def skew_casadi(vector):
    """Returns the skew-symmetric matrix of a vector using CasADi."""
    return ca.vertcat(
        ca.horzcat(0, -vector[2], vector[1]),
        ca.horzcat(vector[2], 0, -vector[0]),
        ca.horzcat(-vector[1], vector[0], 0)
    )

def quat_mult_casadi(q1, q2):
    """Quaternion multiplication using CasADi."""
    w1, x1, y1, z1 = q1[0], q1[1], q1[2], q1[3]
    w2, x2, y2, z2 = q2[0], q2[1], q2[2], q2[3]
    w = w1*w2 - x1*x2 - y1*y2 - z1*z2
    x = w1*x2 + x1*w2 + y1*z2 - z1*y2
    y = w1*y2 - x1*z2 + y1*w2 + z1*x2
    z = w1*z2 + x1*y2 - y1*x2 + z1*w2
    return ca.vertcat(w, x, y, z)

def quat_rotate_casadi(q, v):
    """Rotate vector v by quaternion q using CasADi."""
    q_conj = ca.vertcat(q[0], -q[1], -q[2], -q[3])
    v_quat = ca.vertcat(ca.SX(0), v[0], v[1], v[2])
    v_rot = quat_mult_casadi(quat_mult_casadi(q, v_quat), q_conj)
    return v_rot[1:4]

def quat_mult_numpy(q1, q2):
    """Quaternion multiplication using NumPy."""
    w1, x1, y1, z1 = q1[0], q1[1], q1[2], q1[3]
    w2, x2, y2, z2 = q2[0], q2[1], q2[2], q2[3]
    w = w1*w2 - x1*x2 - y1*y2 - z1*z2
    x = w1*x2 + x1*w2 + y1*z2 - z1*y2
    y = w1*y2 - x1*z2 + y1*w2 + z1*x2
    z = w1*z2 + x1*y2 - y1*x2 + z1*w2
    return np.array([w, x, y, z])

def quat_rotate_numpy(q, v):
    """Rotate vector v by quaternion q using NumPy."""
    q_conj = np.array([q[0], -q[1], -q[2], -q[3]])
    v_quat = np.concatenate(([0], v))
    v_rot = quat_mult_numpy(quat_mult_numpy(q, v_quat), q_conj)
    return v_rot[1:4]

## Dynamics Functions

In [ ]:
# Define the derivative function

def Theta(e):
    return np.array([
        [e[3], -e[2], e[1]],
        [e[2], e[3], -e[0]],
        [-e[1], e[0], e[3]],
        [-e[0], -e[1], -e[2]]
    ])
def Omega(o):
    return np.array([
        [0, o[2], -o[1], o[0]],
        [-o[2], 0, o[0], o[1]],
        [o[1], -o[0], 0, o[2]],
        [-o[0], -o[1], -o[2], 0]
    ])
def Phi(o,I):
    I_inv = np.linalg.inv(I)
    return np.block([[0.5*Omega(o),np.zeros((4,3))],[np.zeros((3,4)),-I_inv@skew(o)@I]])

def Phi_casadi(o, I):
    I_inv = ca.inv(I)
    top_left = 0.5 * Omega(o)
    top_right = ca.SX.zeros(4, 3)
    bottom_left = ca.SX.zeros(3, 4)
    bottom_right = -I_inv @ skew_casadi(o) @ I

    # Build the block matrix using vertcat and horzcat
    return ca.vertcat(
        ca.horzcat(top_left, top_right),
        ca.horzcat(bottom_left, bottom_right)
    )

# Define the state derivative function using CasADi
def rotational_derivative_casadi(state, tau, I):
    quat = state[0:4]
    omega = state[4:7]

    # Quaternion derivative using quaternion multiplication
    omega_quat = ca.vertcat(0, omega[0], omega[1], omega[2])
    quat_dot = 0.5 * quat_mult_casadi(quat, omega_quat)

    # Angular velocity derivative
    omega_dot = ca.solve(I, tau - ca.cross(omega, I @ omega))

    return ca.vertcat(quat_dot, omega_dot)

def rotational_derivative(state, tau, I):
    omega = state[4:]
    phi = Phi(omega, I)
    return phi @ state + np.vstack((np.zeros((4, 3)), np.linalg.inv(I))) @ tau

# Subproblem Functions

## Opt Given Tau

In [ ]:
def opt_given_tau_cvx(tau, num_iter=None):
    delta = 1e-9
    num_steps = len(tau)
    dt = tf / num_steps

    # Define Parameters
    tau_p = cp.Parameter(tau.shape, value=tau)

    # Propagate Attitude and q vectors

    ome = [x0[10:13] for _ in range(num_steps+1)]
    eps = [x0[6:10] for _ in range(num_steps+1)]

        # Position and velocity update using linearized dynamics
    f, f_dot, f_ddot = calculate_orbital_params(mu, a, e, tf)
    A_pos_val = np.array([
        [0, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, 1]
    ])
    A_vel_val = np.array([
        [3 * f_dot**2, 0, 0, 0, 2 * f_dot, 0],
        [0, 0, 0, -2 * f_dot, 0, 0],
        [0, 0, -f_ddot, 0, 0, 0]
    ])
    B_vel_val = np.array([
        [1/m, 0, 0],
        [0, 1/m, 0],
        [0, 0, 1/m]
    ])

    A_pos = cp.Constant(A_pos_val)
    A_vel = cp.Constant(A_vel_val)
    B_vel = cp.Constant(B_vel_val)

    for k in range(num_steps):
        rot_dot = rotational_derivative(np.hstack([eps[k], ome[k]]), tau_p[k].value,I)

        eps[k+1] = eps[k] + dt*rot_dot[0:4]
        eps[k+1] = eps[k+1] / np.linalg.norm(eps[k+1])

        ome[k+1] = ome[k] + dt*rot_dot[4:7]

    qs = [np.zeros((num_steps, 3)) for _ in range(len(rs))]

    for i in range(len(qs)):
        qs[i][0] = rs[i]
        for k in range(num_steps-1):
            qs[i][k+1] = qs[i][k] + dt * skew(ome[k]) @ qs[i][k]
            qs[i][k+1] = np.linalg.norm(rs[i])*qs[i][k+1] / np.linalg.norm(qs[i][k+1])

    # Define Variables
    U = [cp.Variable((num_steps, 3)) for _ in range(len(qs))]
    force = cp.Variable((num_steps, 3))

    r = cp.Variable((num_steps + 1, 3)) # Position
    v = cp.Variable((num_steps + 1, 3)) # Velocity

    # Constraints list
    constraints = []

    # Initial and final conditions
    constraints.append(r[0] == x0[:3])
    constraints.append(v[0] == x0[3:6])
    constraints.append(r[-1] == xf[:3])
    constraints.append(v[-1] == xf[3:6])

    U_stack = cp.vstack(U)
    objective = cp.Minimize(cp.sum_squares(U_stack))

    for i in range(len(U)):
        for k in range(num_steps):
            constraints.append(U[i][k] @ qs[i][k] - np.cos(nu) * (cp.norm(U[i][k]) * cp.norm(qs[i][k])) >= 0)

    for k in range(num_steps):
        constraints.append(force[k] == cp.sum([U[i][k] for i in range(len(U))]))
        constraints.append(tau_p[k] == cp.sum([skew(qs[i][k]) @ U[i][k] for i in range(len(U))]))

        # Dynamics constraints for position and velocity
        constraints.append(r[k + 1] == r[k]+dt*A_pos@cp.hstack([r[k],v[k]]))
        constraints.append(v[k + 1] == v[k]+dt*A_vel@cp.hstack([r[k],v[k]]) + dt * (B_vel @ force[k]))

    prob = cp.Problem(objective, constraints)
    if num_iter is None:
        prob.solve(solver=cp.SCS, verbose=True)
    else:
        prob.solve(solver=cp.SCS, max_iters = num_iter, verbose=True)

    if prob.status in [cp.INFEASIBLE, cp.UNBOUNDED]:
        return "Unsolved X", "Unsolved U", "Unsolved Q", prob

    return np.hstack([r.value, v.value, eps, ome]), [U[i].value for i in range(len(U))], qs, prob.value

In [ ]:
def opt_given_tau_ipopt(tau, num_iter=None):
    num_steps = N
    num_agents = len(rs)
    dt = tf / num_steps

    U_guess = np.zeros((num_agents, num_steps, 3))

    # Convert constants to CasADi types
    I_casadi = ca.DM(I)
    m_casadi = ca.DM(m)
    dt_casadi = dt  # dt is already a scalar, so no need to convert
    epsilon_casadi = epsilon  # Assuming epsilon is a scalar
    nu_casadi = nu  # nu in radians

    # Flatten U into a vector symbolic variable
    U = ca.SX.sym('U', num_agents * num_steps * 3)  # Control inputs

    # Create CasADi variables for states (flattened)
    r = ca.SX.sym('r', (num_steps + 1) * 3)
    v = ca.SX.sym('v', (num_steps + 1) * 3)

    # Define the cost function
    cost = ca.sumsqr(U)  # Sum of squares of control inputs over all agents and timesteps

    # Define constraints
    constraints = []
    constraints_lb = []
    constraints_ub = []

    # Helper functions to index into flattened arrays
    def get_U(i, k):
        idx = (i * num_steps + k) * 3
        return U[idx:idx + 3]

    def get_r(k):
        idx = k * 3
        return r[idx:idx + 3]

    def get_v(k):
        idx = k * 3
        return v[idx:idx + 3]

    # Initial conditions constraints
    constraints.extend([
        get_r(0) - x0[0:3],
        get_v(0) - x0[3:6],
    ])
    constraints_lb.extend([0] * 3 + [0] * 3)
    constraints_ub.extend([0] * 3 + [0] * 3)

    ome = [x0[10:13] for _ in range(num_steps+1)]
    eps = [x0[6:10] for _ in range(num_steps+1)]

    for k in range(num_steps):
        rot_dot = rotational_derivative(np.hstack([eps[k], ome[k]]), tau[k],I)

        eps[k+1] = eps[k] + dt*rot_dot[0:4]
        eps[k+1] = eps[k+1] / np.linalg.norm(eps[k+1])

        ome[k+1] = ome[k] + dt*rot_dot[4:7]

    qs = [np.zeros((num_steps+1, 3)) for _ in range(len(rs))] #! changed here

    for i in range(len(qs)):
        qs[i][0] = rs[i]
        for k in range(num_steps-1):
            qs[i][k+1] = qs[i][k] + dt * skew(ome[k]) @ qs[i][k]
            qs[i][k+1] = np.linalg.norm(rs[i])*qs[i][k+1] / np.linalg.norm(qs[i][k+1])

    # Dynamics equations and constraints
    for k in range(num_steps):
        # Compute torque and force
        torque_curr = ca.SX.zeros(3)
        force_curr = ca.SX.zeros(3)
        for i in range(num_agents):
            Q_ik = ca.DM(qs[i][k])
            U_ik = get_U(i, k)
            torque_curr += skew_casadi(Q_ik) @ U_ik
            force_curr += U_ik

            # Pointing Constraint
            dot_product = ca.dot(U_ik, Q_ik)
            norm_U = smooth_norm(U_ik, epsilon_casadi)
            norm_Q = smooth_norm(Q_ik, epsilon_casadi)
            pointing_constraint = dot_product - ca.cos(nu_casadi) * norm_U * norm_Q
            constraints.append(pointing_constraint)
            constraints_lb.append(0)
            constraints_ub.append(ca.inf)  # No upper bound

        # Torque constraint: sum_i skew(Q_i) * U_i == tau
        constraints.append(torque_curr - tau[k])
        constraints_lb.extend([0] * 3)
        constraints_ub.extend([0] * 3)

        # Orbital parameters
        f, f_dot, f_ddot = calculate_orbital_params(mu, a, e, k * dt)

        A_vel = ca.SX.zeros(3, 6)
        A_vel[0, :] = ca.horzcat(3 * f_dot ** 2, 0, 0, 0, 2 * f_dot, 0)
        A_vel[1, :] = ca.horzcat(0, 0, 0, -2 * f_dot, 0, 0)
        A_vel[2, :] = ca.horzcat(0, 0, -f_ddot, 0, 0, 0)
        B_vel = (1 / m) * ca.SX_eye(3)

        # State updates for position and velocity
        r_k = get_r(k)
        v_k = get_v(k)
        r_next = r_k + dt * v_k
        v_next = v_k + dt * (A_vel @ ca.vertcat(r_k, v_k) + B_vel @ force_curr)

        # Append dynamics constraints for position and velocity
        constraints.extend([
            get_r(k + 1) - r_next,
            get_v(k + 1) - v_next,
        ])
        constraints_lb.extend([0] * 3 + [0] * 3)
        constraints_ub.extend([0] * 3 + [0] * 3)


    # Final state constraints for position and velocity
    constraints.extend([
        get_r(num_steps) - xf[0:3],
        get_v(num_steps) - xf[3:6],
    ])
    constraints_lb.extend([0] * 3 + [0] * 3)
    constraints_ub.extend([0] * 3 + [0] * 3)

    # Flatten constraints
    g = ca.vertcat(*constraints)
    lbg = np.array(constraints_lb)
    ubg = np.array(constraints_ub)

    # Decision variables
    opt_vars = ca.vertcat(U, r, v)

    # Initial guess
    opt_vars_init = np.concatenate([
        U_guess.flatten(),
        np.tile(x0[0:3], num_steps + 1),
        np.tile(x0[3:6], num_steps + 1),
    ])

    # Define NLP problem
    nlp = {'x': opt_vars, 'f': cost, 'g': g}

    if num_iter is None:
        opts = {'ipopt': {'print_level': 5}}
    else:
        opts = {'ipopt': {'max_iter': num_iter, 'print_level': 5}}

    # Create solver instance
    solver = ca.nlpsol('solver', 'ipopt', nlp, opts)

    original_stdout = sys.stdout

    # Solve the optimization problem
    try:
        # Open os.devnull to redirect stdout
        with open(os.devnull, 'w') as f:
            #sys.stdout = f  # Redirect stdout to /dev/null
            # Run the solver
            sol = solver(x0=opt_vars_init, lbg=lbg, ubg=ubg)

    finally:
        # Restore stdout to its original state
        sys.stdout = original_stdout

    # Extract the solution
    w_opt = sol['x'].full().flatten()

    # Extract optimized variables
    num_U = num_agents * num_steps * 3
    num_r = (num_steps + 1) * 3
    num_v = (num_steps + 1) * 3

    idx = 0
    U_opt = w_opt[idx:idx + num_U].reshape(num_agents, num_steps, 3)
    idx += num_U
    r_opt = w_opt[idx:idx + num_r].reshape(num_steps + 1, 3)
    idx += num_r
    v_opt = w_opt[idx:idx + num_v].reshape(num_steps + 1, 3)

    # Convert lists to arrays
    eps_opt = np.array(eps)
    ome_opt = np.array(ome)
    Q_opt = np.array(qs)

    X_opt = np.hstack([r_opt, v_opt, eps_opt, ome_opt])

    return X_opt, U_opt, Q_opt, np.sum(np.square(U_opt))

## Full NLP Solver

In [ ]:
def full_nlp(time_limit=300,U_guess=None, X_guess=None, num_iter=None, pointing=False):
    num_steps = N
    num_agents = len(rs)
    dt = tf / num_steps

    if U_guess is None:
        U_guess = np.zeros((num_agents, num_steps, 3))

    if X_guess is None:
        opt_vars_init = np.concatenate([
            U_guess.flatten(),
            np.tile(x0[0:3], num_steps+1),
            np.tile(x0[3:6], num_steps+1),
            np.tile(x0[6:10], num_steps+1),
            np.tile(x0[10:13], num_steps+1),
            np.tile(np.array(rs).flatten(), num_steps+1),
        ])
    else:
        opt_vars_init = np.concatenate([
            U_guess.flatten(),
            X_guess.flatten()
        ])


    # Normalize initial and final quaternions
    x0[6:10] = x0[6:10] / np.linalg.norm(x0[6:10])
    xf[6:10] = xf[6:10] / np.linalg.norm(xf[6:10])

    # Convert constants to CasADi types
    I_casadi = ca.DM(I)
    m_casadi = ca.DM(m)
    dt_casadi = ca.DM(dt)

    # Flatten U into a vector symbolic variable
    U = ca.SX.sym('U', num_agents * num_steps * 3)  # Control inputs

    # Create CasADi variables for states (flattened)
    r = ca.SX.sym('r', (num_steps+1) * 3)
    v = ca.SX.sym('v', (num_steps+1) * 3)
    eps = ca.SX.sym('eps', (num_steps+1) * 4)
    ome = ca.SX.sym('ome', (num_steps+1) * 3)
    Q = ca.SX.sym('Q', num_agents * (num_steps+1) * 3)

    # Define the cost function
    cost = ca.sumsqr(U)  # Sum of squares of control inputs over all agents and timesteps

    # Define constraints
    constraints = []
    constraints_lb = []
    constraints_ub = []

    # Helper functions to index into flattened arrays
    def get_U(i, k):
        idx = (i * num_steps + k) * 3
        return U[idx:idx+3]

    def get_r(k):
        idx = k * 3
        return r[idx:idx+3]

    def get_v(k):
        idx = k * 3
        return v[idx:idx+3]

    def get_eps(k):
        idx = k * 4
        return eps[idx:idx+4]

    def get_ome(k):
        idx = k * 3
        return ome[idx:idx+3]

    def get_Q(i, k):
        idx = (i * (num_steps+1) + k) * 3
        return Q[idx:idx+3]

    # Initial conditions constraints
    constraints.extend([
        get_r(0) - x0[0:3],
        get_v(0) - x0[3:6],
        get_eps(0) - x0[6:10],
        get_ome(0) - x0[10:13],
    ])
    constraints_lb.extend([0]*3 + [0]*3 + [0]*4 + [0]*3)
    constraints_ub.extend([0]*3 + [0]*3 + [0]*4 + [0]*3)

    for i in range(num_agents):
        constraints.append(get_Q(i, 0) - rs[i])
        constraints_lb.extend([0]*3)
        constraints_ub.extend([0]*3)

    # Dynamics equations and constraints
    #torque_curr = ca.SX.zeros(3)
    for k in range(num_steps):
        # Compute torque and force
        #torque_prev = torque_curr
        torque_curr = ca.SX.zeros(3)
        force_curr = ca.SX.zeros(3)
        for i in range(num_agents):
            Q_ik = get_Q(i, k)
            U_ik = get_U(i, k)
            torque_curr += skew_casadi(Q_ik) @ U_ik
            force_curr += U_ik

        # Orbital parameters (implement your actual function)
        f, f_dot, f_ddot = calculate_orbital_params(mu, a, e, k * dt_casadi)

        A_vel = ca.SX.zeros(3,6)
        A_vel[0,:] = ca.horzcat(3 * f_dot**2, 0, 0, 0, 2 * f_dot, 0)
        A_vel[1,:] = ca.horzcat(0, 0, 0, -2 * f_dot, 0, 0)
        A_vel[2,:] = ca.horzcat(0, 0, -f_ddot, 0, 0, 0)
        B_vel = (1/m_casadi) * ca.SX.eye(3)

        # State updates
        r_k = get_r(k)
        v_k = get_v(k)
        r_next = r_k + dt_casadi * v_k
        v_next = v_k + dt_casadi * (A_vel @ ca.vertcat(r_k, v_k) + B_vel @ force_curr)

        # Orientation and angular velocity update using original dynamics method
        eps_k = get_eps(k)
        ome_k = get_ome(k)

        phi = Phi_casadi(ome_k, I_casadi)

        eps_next = eps_k + (dt_casadi * phi @ ca.vertcat(eps_k, ome_k))[0:4]
        eps_next = eps_next / ca.norm_2(eps_next)
        ome_next = ome_k + (dt_casadi * phi @ ca.vertcat(eps_k, ome_k))[4:7] + dt_casadi * (ca.inv(I_casadi) @ torque_curr)

        # Append dynamics constraints
        constraints.extend([
            get_r(k+1) - r_next,
            get_v(k+1) - v_next,
            get_eps(k+1) - eps_next,
            get_ome(k+1) - ome_next,
        ])
        constraints_lb.extend([0]*3 + [0]*3 + [0]*4 + [0]*3)
        constraints_ub.extend([0]*3 + [0]*3 + [0]*4 + [0]*3)

        #tau_diff = torque_curr - torque_prev
        #smoothness_constraint = ca.sumsqr(tau_diff)  # (tau[k+1] - tau[k])^2
        #constraints.append(smoothness_constraint)
        #constraints_lb.append(0)  # Lower bound for the smoothness constraint
        #constraints_ub.append(10)  # Upper bound for the smoothness constraint

        # Update Q and add pointing constraints
        for i in range(num_agents):
            Q_ik = get_Q(i, k)
            Q_next = Q_ik + dt_casadi * skew_casadi(ome_k) @ Q_ik
            Q_next = Q_next / ca.norm_2(Q_next) * ca.norm_2(rs[i])  # Rescale Q

            constraints.append(get_Q(i, k+1) - Q_next)
            constraints_lb.extend([0]*3)
            constraints_ub.extend([0]*3)

            if pointing:
                # Pointing constraint
                U_ik = get_U(i, k)

                # Compute the dot product
                dot_product = ca.dot(U_ik, Q_ik)

                # Compute the norms using smooth_norm
                norm_U = smooth_norm(U_ik, epsilon)
                norm_qs = smooth_norm(Q_ik, epsilon)

                # Compute the cosine of the angle
                cos_nu = ca.cos(nu)

                # Constraint expression
                pointing_constraint = dot_product - cos_nu * norm_U * norm_qs

                # Add the constraint to the list
                constraints.append(pointing_constraint)
                constraints_lb.append(0)
                constraints_ub.append(ca.inf)  # No upper bound

    # Final state constraints
    constraints.extend([
        get_r(num_steps) - xf[0:3],
        get_v(num_steps) - xf[3:6],
        get_eps(num_steps) - xf[6:10],
        get_ome(num_steps) - xf[10:13],
    ])
    constraints_lb.extend([0]*3 + [0]*3 + [0]*4 + [0]*3)
    constraints_ub.extend([0]*3 + [0]*3 + [0]*4 + [0]*3)

    # Flatten constraints
    g = ca.vertcat(*constraints)
    lbg = np.array(constraints_lb)
    ubg = np.array(constraints_ub)

    # Decision variables
    opt_vars = ca.vertcat(U, r, v, eps, ome, Q)

    # Define NLP problem
    nlp = {'x': opt_vars, 'f': cost, 'g': g}

    if num_iter is None:
        opts = {'ipopt': {'print_level': 5}}
    else:
        opts = {'ipopt': {'max_iter': num_iter, 'print_level': 5}}
    if time_limit is not None:
        opts = {'ipopt': {'max_wall_time': float(time_limit), 'max_cpu_time': float(time_limit),'max_iter': 1e9 }}
    # Create solver instance
    solver = ca.nlpsol('solver', 'ipopt', nlp, opts)

    # Solve the problem
    sol = solver(x0=opt_vars_init, lbg=lbg, ubg=ubg)

    # Extract the solution
    w_opt = sol['x'].full().flatten()
    status = solver.stats()
    ret = status["return_status"]
    suc = status["success"]

    # Extract optimized variables
    num_U = num_agents * num_steps * 3
    num_r = (num_steps+1) * 3
    num_v = (num_steps+1) * 3
    num_eps = (num_steps+1) * 4
    num_ome = (num_steps+1) * 3
    num_Q = num_agents * (num_steps+1) * 3

    idx = 0
    U_opt = w_opt[idx:idx+num_U].reshape(num_agents, num_steps, 3)
    idx += num_U
    r_opt = w_opt[idx:idx+num_r].reshape(num_steps+1, 3)
    idx += num_r
    v_opt = w_opt[idx:idx+num_v].reshape(num_steps+1, 3)
    idx += num_v
    eps_opt = w_opt[idx:idx+num_eps].reshape(num_steps+1, 4)
    idx += num_eps
    ome_opt = w_opt[idx:idx+num_ome].reshape(num_steps+1, 3)
    idx += num_ome
    Q_opt = w_opt[idx:idx+num_Q].reshape(num_agents, num_steps+1, 3)

    X_opt = np.hstack([r_opt, v_opt, eps_opt, ome_opt])

    return U_opt, X_opt, Q_opt, ret, suc


## Tau Projection Non-Linear

In [ ]:
# Define the cost function and constraints for multiple shooting
def tau_proj_nonlin(tau_hist, num_iter=None):
    num_steps = N  # Number of steps
    dt = tf / num_steps

    # Convert constants to CasADi types
    I_casadi = ca.DM(I)
    dt_casadi = ca.DM(dt)
    epsilon_casadi = ca.DM(epsilon)

    # Define CasADi variables for control inputs and states
    tau = [ca.SX.sym(f'tau_{k}', 3) for k in range(num_steps)]  # Control torques
    state = [ca.SX.sym(f'state_{k}', 7) for k in range(num_steps + 1)]  # State (quaternion + angular velocity)

    cost = 0

    for k in range(num_steps):
        cost += ca.sumsqr(tau[k]-tau_hist[k])  # Minimize control effort

    # Initialize constraints list
    constraints = []
    lbg = []
    ubg = []

    # Initial state constraint (state[0] == a0)
    constraints.append(state[0] - a0)
    lbg.extend([0]*7)
    ubg.extend([0]*7)

    for k in range(num_steps):
        state_k = state[k]
        tau_k = tau[k]

        # Extract the quaternion and angular velocity from the current state
        eps_k = state_k[0:4]   # Quaternion part
        ome_k = state_k[4:7]   # Angular velocity part

        # Calculate the phi matrix for current angular velocity
        phi = Phi_casadi(ome_k, I_casadi)

        # Propagate quaternion and angular velocity using the provided dynamics
        rotational_update = phi @ ca.vertcat(eps_k, ome_k)
        eps_next = eps_k + dt_casadi * rotational_update[0:4]  # Quaternion update
        ome_next = ome_k + dt_casadi * rotational_update[4:7] + dt_casadi * (ca.inv(I_casadi) @ tau_k)  # Angular velocity update

        # Normalize the quaternion using smooth norm
        quat_next = eps_next / smooth_norm(eps_next, epsilon_casadi)

        # Combine the normalized quaternion and updated angular velocity
        state_k_next_normalized = ca.vertcat(quat_next, ome_next)

        # Append the dynamics constraint (ensure continuity between intervals)
        constraints.append(state[k + 1] - state_k_next_normalized)
        lbg.extend([0] * 7)
        ubg.extend([0] * 7)

                # Smoothness constraint: |tau[k+1] - tau[k]|^2 < epsilon
        #if k < num_steps-1:
        #    tau_diff = tau[k+1] - tau[k]
        #    smoothness_constraint = ca.sumsqr(tau_diff)  # (tau[k+1] - tau[k])^2
        #    constraints.append(smoothness_constraint)
        #    lbg.append(0)  # Lower bound for the smoothness constraint
        #    ubg.append(1)  # Upper bound for the smoothness constraint

    # Final state constraint (state[num_steps] == af)
    constraints.append(state[num_steps] - af)
    lbg.extend([0]*7)
    ubg.extend([0]*7)

    # Set up the optimization problem
    opt_vars = ca.vertcat(*tau, *state)  # Decision variables (controls + states)
    g = ca.vertcat(*constraints)  # All constraints

    # Convert bounds lists to NumPy arrays
    lbg = np.array(lbg)
    ubg = np.array(ubg)

    # Set bounds for tau (optional: if you want to limit tau further)
    tau_lower_bound = -ca.inf * np.ones(num_steps * 3)
    tau_upper_bound = ca.inf * np.ones(num_steps * 3)

    # For state variables, we leave them unbounded
    state_lower_bound = -ca.inf * np.ones((num_steps + 1) * 7)  # 7 state variables per step
    state_upper_bound = ca.inf * np.ones((num_steps + 1) * 7)

    # Combine bounds for tau and state variables
    lbx = np.concatenate([tau_lower_bound, state_lower_bound])
    ubx = np.concatenate([tau_upper_bound, state_upper_bound])

    # Define the NLP problem
    nlp = {'x': opt_vars, 'f': cost, 'g': g}

    if num_iter is None:
        opts = {
            'ipopt': {
                'print_level': 4,  # Solver verbosity level
            }
        }
    else:
        opts = {
            'ipopt': {
                'max_iter': num_iter,
                'print_level': 4,  # Solver verbosity level
            }
        }

    # Create a solver instance with IPOPT
    solver = ca.nlpsol('solver', 'ipopt', nlp, opts)

            # Store the original stdout
    original_stdout = sys.stdout

    # Define initial guess for tau and state
    tau_init_guess = np.zeros(num_steps * 3)

    # Set initial state guess based on linear interpolation between a0 and af
    state_init_guess = np.linspace(a0, af, num_steps + 1)

    # Flatten the initial guess
    state_init_guess = state_init_guess.flatten()

    # Combine the initial guesses
    x0 = np.concatenate([tau_init_guess, state_init_guess])

    # Solve the optimization problem
    try:
        # Open os.devnull to redirect stdout
        with open(os.devnull, 'w') as f:
            sys.stdout = f  # Redirect stdout to /dev/null
            # Run the solver
            sol = solver(x0=x0, lbg=lbg, ubg=ubg)

    finally:
        # Restore stdout to its original state
        sys.stdout = original_stdout

    # Extract the solution
    w_opt = sol['x'].full().flatten()

    # Split the solution into tau and state
    tau_opt = w_opt[:num_steps * 3].reshape(num_steps, 3)  # Optimized torques
    state_opt = w_opt[num_steps * 3:].reshape(num_steps + 1, 7)  # Optimized states

    return tau_opt, state_opt

## Particle Swarm Optimization

In [ ]:
def pso_function(tau_flat):
    costs = []
    for candidate_tau_flat in tau_flat:
        # Reshape the candidate tau from a flat array into a 3xN matrix
        candidate_tau = candidate_tau_flat.reshape(N, 3)

        # Call your opt_given_tau function with the reshaped tau

        fitted_tau = tau_proj_nonlin(candidate_tau)[0]
        opt_result = opt_given_tau_ipopt(fitted_tau, num_iter=1000)
        cost = opt_result[3]
        fin_ang_state = np.array(opt_result[0][-1][6:13])
        alpha=1e4
        print("Current Cost: " + str(cost))

        if cost == 0:
          cost = np.inf

        costs.append(cost+(alpha*np.linalg.norm(fin_ang_state-xf[6:13]))**2)

    return np.array(costs)

## Genetic Algorithm Functions

In [ ]:
def pop_gen(a0, af, tf, I, pop_size=50, num_iter=None):
    num_steps = N  # Number of steps
    dt = tf / num_steps

    # Convert constants to CasADi types
    I_casadi = ca.DM(I)
    dt_casadi = ca.DM(dt)
    epsilon_casadi = ca.DM(epsilon)

    # Define the cost function and constraints for multiple shooting
    def multiple_shooting_optimization(a0, af):
        # Define CasADi variables for control inputs and states
        tau = [ca.SX.sym(f'tau_{k}', 3) for k in range(num_steps)]  # Control torques
        state = [ca.SX.sym(f'state_{k}', 7) for k in range(num_steps + 1)]  # State (quaternion + angular velocity)
        cost = 0

        for k in range(num_steps):
            cost += ca.sumsqr(tau[k])  # Minimize control effort

        # Initialize constraints list
        constraints = []
        lbg = []
        ubg = []

        # Initial state constraint (state[0] == a0)
        constraints.append(state[0] - a0)
        lbg.extend([0]*7)
        ubg.extend([0]*7)

        # Dynamics constraints using multiple shooting
        for k in range(num_steps):
            state_k = state[k]
            tau_k = tau[k]

            # Propagate the state to the next step using Euler integration
            state_k_next = state_k + dt_casadi * rotational_derivative_casadi(state_k, tau_k, I_casadi)

            # Normalize the quaternion using smooth norm
            quat_next = state_k_next[0:4] / smooth_norm(state_k_next[0:4], epsilon_casadi)
            state_k_next_normalized = ca.vertcat(quat_next, state_k_next[4:7])

            # Append the dynamics constraint (ensure continuity between intervals)
            constraints.append(state[k+1] - state_k_next_normalized)
            lbg.extend([0]*7)
            ubg.extend([0]*7)

        # Final state constraint (state[num_steps] == af)
        constraints.append(state[num_steps] - af)
        lbg.extend([0]*7)
        ubg.extend([0]*7)

        # Set up the optimization problem
        opt_vars = ca.vertcat(*tau, *state)  # Decision variables (controls + states)
        g = ca.vertcat(*constraints)  # All constraints

        # Convert bounds lists to NumPy arrays
        lbg = np.array(lbg)
        ubg = np.array(ubg)

        # Set bounds for tau (optional: if you want to limit tau further)
        tau_lower_bound = -ca.inf * np.ones(num_steps * 3)  # Set torque lower bound to -100
        tau_upper_bound = ca.inf * np.ones(num_steps * 3)   # Set torque upper bound to 100

        ## For state variables, we leave them unbounded
        state_lower_bound = -ca.inf * np.ones((num_steps + 1) * 7)  # 7 state variables per step
        state_upper_bound = ca.inf * np.ones((num_steps + 1) * 7)

        # Combine bounds for tau and state variables
        lbx = np.concatenate([tau_lower_bound, state_lower_bound])
        ubx = np.concatenate([tau_upper_bound, state_upper_bound])

        # Define the NLP problem
        nlp = {'x': opt_vars, 'f': cost, 'g': g}

        if num_iter is None:
            opts = {
                'ipopt': {
                    'max_iter': num_iter,
                    'print_level': 0,  # Solver verbosity level
                }
            }
        else:
            opts = {
                'ipopt': {
                    'print_level': 0,  # Solver verbosity level
                }
            }



        # Define initial guess for tau and state
        tau_init_guess = np.random.uniform(-1e-6, 1e-6, num_steps * 3)  # Random initial guess for tau

        # Set initial state guess based on linear interpolation between a0 and af
        state_init_guess = np.linspace(a0, af, num_steps + 1)

        # Flatten the initial guess
        state_init_guess = state_init_guess.flatten()

        # Combine the initial guesses
        x0 = np.concatenate([tau_init_guess, state_init_guess])

        original_stdout = sys.stdout
        # Create solver instance
        solver = ca.nlpsol('solver', 'ipopt', nlp, opts)

        # Run the solver

        try:
          # Open os.devnull to redirect stdout
          with open(os.devnull, 'w') as f:
            sys.stdout = f  # Redirect stdout to /dev/null
            sol = solver(x0=x0, lbg=lbg, ubg=ubg, ubx=ubx, lbx=lbx)
        finally:
          # Restore the original stdout
          sys.stdout = original_stdout

        # Extract the solution
        w_opt = sol['x'].full().flatten()

        # Split the solution into tau and state
        tau_opt = w_opt[:num_steps * 3].reshape(num_steps, 3)  # Optimized torques
        state_opt = w_opt[num_steps * 3:].reshape(num_steps + 1, 7)  # Optimized states

        return tau_opt, state_opt

    # Generate a population of feasible solutions
    population = []
    for _ in range(pop_size):
        # Solve the multiple-shooting optimization for each individual in the population
        tau_opt, state_opt = multiple_shooting_optimization(a0, af)
        population.append([tau_opt, state_opt])

    return population


In [ ]:
# Define your fitness function
def fitness_func(ga_instance, solution, solution_idx):
    # Reshape the solution into the correct tau format (num_steps x 3)
    tau = solution.reshape((N, 3))
    tau = tau_proj_nonlin(tau)[0]
    opt_result = opt_given_tau_ipopt(tau, num_iter=3000) #changed from 1000 - 3000

    cost = opt_result[3]
    fin_ang_state = np.array(opt_result[0][-1][6:13])
    alpha=1e4

    #fitness = 1 / (cost + (alpha*np.linalg.norm(fin_ang_state-xf[6:13]))**2) #any particular reason for this choice in fitness function?
    fitness = 1 / (cost)

    print("Current Cost: " + str(cost))
    if fitness == np.inf:
      fitness = 0
    return fitness

In [ ]:
def genetic_algorithm_optimization(time_limit = None, pop_size=50, num_gens=50, warm_init=True):
    time_start = time.perf_counter()
      # Progress indicator callback
    def callback_generation(ga_instance):
        best_overall_fitness = ga_instance.best_solution()[1]
        if time_limit is not None:
            if (time.perf_counter() - time_start) >= time_limit:
                # optional: print a final line right before stopping
                print(f"Time limit reached. Stopping at generation {ga_instance.generations_completed}. "
                      f"Best fitness = {best_overall_fitness}")
                return "stop"
            # Print the best fitness values
        # Print the best fitness values
        print(f"Generation {ga_instance.generations_completed}: Best Overall Fitness = {best_overall_fitness}")

    if warm_init:
                # Generate the initial population using pop_gen
        initial_population_data = pop_gen(x0[6:13], xf[6:13], tf, I, pop_size=pop_size)

        # Extract only the tau values (which are control torques) from the population
        initial_population = [individual[0].flatten() for individual in initial_population_data]

        ga_instance = pygad.GA(
            num_generations=num_gens,
            num_parents_mating=int(pop_size/2),
            initial_population=initial_population,
            fitness_func=fitness_func,
            num_genes=N * 3,  # Number of elements in each solution (tau vectors)
            mutation_percent_genes=10,  # Mutation percentage
            crossover_type="two_points",
            mutation_type="random",  # Random mutation
            mutation_by_replacement=False,
            parent_selection_type="sss",
            keep_parents=-1,
            keep_elitism= int(pop_size/4),
            allow_duplicate_genes=True,  # Prevent duplicates
            on_generation=callback_generation  # Add the progress callback here
        )
    else:
        ga_instance = pygad.GA(
            num_generations=num_gens,
            num_parents_mating=int(pop_size/2),
            initial_population=None,
            sol_per_pop=pop_size,
            fitness_func=fitness_func,
            num_genes=N * 3,  # Number of elements in each solution (tau vectors)
            mutation_percent_genes=1,  # Mutation percentage
            crossover_type="two_points",
            mutation_type="random",  # Random mutation
            mutation_by_replacement=False,
            parent_selection_type="sss",  # Rank-based selection
            keep_parents=-1,
            keep_elitism= int(pop_size/4),
            allow_duplicate_genes=True,  # Prevent duplicates
            init_range_low=-1e-9,
            init_range_high=1e-9,
            on_generation=callback_generation  # Add the progress callback here
        )

        # Run the genetic algorithm
    start_time = time.time() # Run the genetic algorithm
    ga_instance.run()
    end_time = time.time()
    ga_time_taken = end_time-start_time

    # After running, get the best solution
    best_solution, best_solution_fitness, best_solution_idx = ga_instance.best_solution()
    print("Finished GA")
    print(f"Time taken inside of GA: {ga_time_taken}")
    print("Best solution fitness:", best_solution_fitness)
    print(f"Generations Completed : {ga_instance.generations_completed}")

    # Reshape the best solution back to the tau format (num_steps x 3)
    tau_opt = best_solution.reshape((N, 3))

    # Return the best solution tau_opt
    return tau_opt, ga_time_taken, ga_instance.generations_completed

## Forward Pass of Dynamics

In [ ]:
def forward_pass_dynamics(U):
    num_steps = N
    num_agents = len(rs)
    dt = tf / num_steps

    U = np.array(U)

    # Initialize state vectors
    r_fin = np.zeros((num_steps + 1, 3))
    v_fin = np.zeros((num_steps + 1, 3))
    eps_fin = np.zeros((num_steps + 1, 4))
    ome_fin = np.zeros((num_steps + 1, 3))
    Q_fin = np.zeros((num_agents, num_steps + 1, 3))

    # Set initial conditions
    for i in range(num_agents):
        Q_fin[i][0] = rs[i]
    r_fin[0] = x0[0:3]
    v_fin[0] = x0[3:6]
    eps_fin[0] = x0[6:10] / np.linalg.norm(x0[6:10])  # Ensure initial quaternion is normalized
    ome_fin[0] = x0[10:13]

    # Forward pass through the dynamics using provided U
    for k in range(num_steps):
        # Compute current torque and force
        torque_curr = np.sum([skew(Q_fin[i][k]) @ U[i][k] for i in range(num_agents)], axis=0)
        force_curr = np.sum([U[i][k] for i in range(num_agents)], axis=0)

        # Position and velocity update using linearized dynamics
        f, f_dot, f_ddot = calculate_orbital_params(mu, a, e, k * dt)
        A_pos = np.array([
            [0, 0, 0, 1, 0, 0],
            [0, 0, 0, 0, 1, 0],
            [0, 0, 0, 0, 0, 1]
        ])
        A_vel = np.array([
            [3 * f_dot**2, 0, 0, 0, 2 * f_dot, 0],
            [0, 0, 0, -2 * f_dot, 0, 0],
            [0, 0, -f_ddot, 0, 0, 0]
        ])
        B_vel = np.array([
            [1/m, 0, 0],
            [0, 1/m, 0],
            [0, 0, 1/m]
        ])

        # Dynamics update for position and velocity
        r_fin[k + 1] = r_fin[k] + dt * A_pos @ np.hstack([r_fin[k], v_fin[k]])
        v_fin[k + 1] = v_fin[k] + dt * A_vel @ np.hstack([r_fin[k], v_fin[k]]) + dt * (B_vel @ force_curr)

        # Update orientation (eps) and angular velocity (ome)
        phi = Phi(ome_fin[k], I)
        rotational_update = phi @ np.hstack([eps_fin[k], ome_fin[k]])
        eps_next = eps_fin[k] + dt * rotational_update[0:4]
        ome_next = ome_fin[k] + dt * rotational_update[4:] + dt * (np.linalg.inv(I) @ torque_curr)

        # Normalize the quaternion part of eps_next
        eps_next /= np.linalg.norm(eps_next)

        # Store the updated quaternion and angular velocity
        eps_fin[k + 1] = eps_next
        ome_fin[k + 1] = ome_next

        # Update Q for each agent based on angular velocity
        for i in range(num_agents):
            Q_ik = Q_fin[i][k]
            Q_next = Q_ik + dt * skew(ome_fin[k]) @ Q_ik
            Q_next = Q_next / np.linalg.norm(Q_next) * np.linalg.norm(rs[i])  # Normalize to original magnitude
            Q_fin[i][k + 1] = Q_next

    # Return the final states (position, velocity, orientation, angular velocity) and Q values
    X_opt = np.hstack([r_fin, v_fin, eps_fin, ome_fin])

    return X_opt, Q_fin


# Plotting Functions

In [ ]:
def plot_att_vs_time(state_opt, tf):
    if isinstance(state_opt, list):
        num_steps = len(state_opt) - 1
    else:
        num_steps = state_opt.shape[0] - 1

    time = np.linspace(0, tf, num_steps + 1)

    plt.figure(figsize=(12, 8))

    # Plot quaternion components (first 4 state variables)
    plt.subplot(2, 1, 1)
    plt.plot(time, state_opt[:, 0], label=r'$\epsilon_1$')
    plt.plot(time, state_opt[:, 1], label=r'$\epsilon_2$')
    plt.plot(time, state_opt[:, 2], label=r'$\epsilon_3$')
    plt.plot(time, state_opt[:, 3], label=r'$\epsilon_4$')
    plt.title('Quaternion Components vs Time')
    plt.xlabel('Time [s]')
    plt.ylabel('Quaternion Components')
    plt.legend()

    # Plot angular velocities (last 3 state variables)
    plt.subplot(2, 1, 2)
    plt.plot(time, state_opt[:, 4], label=r'$\omega_1$')
    plt.plot(time, state_opt[:, 5], label=r'$\omega_2$')
    plt.plot(time, state_opt[:, 6], label=r'$\omega_3$')
    plt.title('Angular Velocities vs Time')
    plt.xlabel('Time [s]')
    plt.ylabel('Angular Velocity [rad/s]')
    plt.legend()

    plt.tight_layout()
    plt.show()


def plot_qs_vs_time(qs_hist_updated, tf):
    num_steps = len(qs_hist_updated[0])
    time = np.linspace(0, tf, num_steps)

    qs_hist_updated = np.array(qs_hist_updated)  # Convert list of qs to a numpy array

    plt.figure(figsize=(12, 8))

    # Plot each component of q against time
    for i in range(len(qs_hist_updated)):
        plt.plot(time, qs_hist_updated[i], label=f'q{i+1}')

    plt.title('q Components vs Time')
    plt.xlabel('Time [s]')
    plt.ylabel('q Components')
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_pos_vel_vs_time(state_opt, tf):
    if isinstance(state_opt, list):
        num_steps = len(state_opt) - 1
    else:
        num_steps = state_opt.shape[0] - 1

    time = np.linspace(0, tf, num_steps + 1)

    plt.figure(figsize=(12, 8))

    # Plot position components (first 3 state variables: x, y, z)
    plt.subplot(2, 1, 1)
    plt.plot(time, state_opt[:, 0], label='x')
    plt.plot(time, state_opt[:, 1], label='y')
    plt.plot(time, state_opt[:, 2], label='z')
    plt.title('Position Components vs Time')
    plt.xlabel('Time [s]')
    plt.ylabel('Position [m]')
    plt.legend()

    # Plot velocity components (next 3 state variables: vx, vy, vz)
    plt.subplot(2, 1, 2)
    plt.plot(time, state_opt[:, 3], label='vx')
    plt.plot(time, state_opt[:, 4], label='vy')
    plt.plot(time, state_opt[:, 5], label='vz')
    plt.title('Velocity Components vs Time')
    plt.xlabel('Time [s]')
    plt.ylabel('Velocity [m/s]')
    plt.legend()

    plt.tight_layout()
    plt.show()

def plot_control_forces_vs_time(U_opt, tf):
    num_agents = len(U_opt)
    num_steps = U_opt[0].shape[0]

    time = np.linspace(0, tf, num_steps)

    plt.figure(figsize=(15, 12))

    # Plot control forces in the x direction for all agents
    plt.subplot(3, 1, 1)
    for i in range(num_agents):
        plt.plot(time, U_opt[i][:, 0], label=f'Agent {i+1}')
    plt.title('Control Force in X Direction vs Time')
    plt.xlabel('Time [s]')
    plt.ylabel('Control Force in X [N]')
    plt.legend()
    plt.grid(True)

    # Plot control forces in the y direction for all agents
    plt.subplot(3, 1, 2)
    for i in range(num_agents):
        plt.plot(time, U_opt[i][:, 1], label=f'Agent {i+1}')
    plt.title('Control Force in Y Direction vs Time')
    plt.xlabel('Time [s]')
    plt.ylabel('Control Force in Y [N]')
    plt.legend()
    plt.grid(True)

    # Plot control forces in the z direction for all agents
    plt.subplot(3, 1, 3)
    for i in range(num_agents):
        plt.plot(time, U_opt[i][:, 2], label=f'Agent {i+1}')
    plt.title('Control Force in Z Direction vs Time')
    plt.xlabel('Time [s]')
    plt.ylabel('Control Force in Z [N]')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

def plot_cost_vs_iteration(cost_history, window_size=5, show_trend_line=True):
    """
    Plots the cost vs. iteration with optional moving average trend line.

    Parameters:
    cost_history (list): List of cost values from each iteration.
    window_size (int): Window size for moving average. Default is 5.
    show_trend_line (bool): Whether to show a linear trend line. Default is True.
    """
    iterations = np.arange(1, len(cost_history) + 1)

    plt.figure(figsize=(10, 6))
    plt.plot(iterations, cost_history, '-o', color='b', label='Cost')

    # Plot moving average trend line
    if window_size > 1:
        moving_avg = uniform_filter1d(cost_history, size=window_size)
        plt.plot(iterations, moving_avg, color='r', linestyle='--', label=f'Moving Avg (window={window_size})')

    # Plot linear regression trend line
    if show_trend_line:
        z = np.polyfit(iterations, cost_history, 1)
        p = np.poly1d(z)
        plt.plot(iterations, p(iterations), color='g', linestyle=':', label='Linear Trend Line')

    plt.title('Cost vs. Iteration')
    plt.xlabel('Iteration')
    plt.ylabel('Cost')
    plt.grid(True)
    plt.legend()
    plt.show()

# Main Functions

### Opt Runtime Measurement

In [ ]:
def measure_opt_runtime(num_runs=30, subset_size=5):
    total_time = 0

    for _ in range(num_runs):
        # Randomly select a subset of 3D arrays from rsfull
        rs_indices = np.random.choice(len(rsfull), subset_size, replace=False)
        rs = [rsfull[i] for i in rs_indices]
        testtau = pop_gen(x0[6:13], xf[6:13], tf, I, pop_size=1, num_iter=100)[0][0]

        # Measure the time taken to run the function
        start_time = time.time()
        opt_given_tau_ipopt(testtau)
        end_time = time.time()

        # Accumulate the time taken
        total_time += (end_time - start_time)

    # Calculate the average time
    avg_time = total_time / num_runs
    return avg_time

### Some Test Code

In [ ]:
testtau = tau_proj_nonlin(np.zeros((N,3)))[0]
print("I got here")
testresults = opt_given_tau_ipopt(testtau)

plot_att_vs_time(testresults[0].T[6:13].T, tf)
plot_pos_vel_vs_time(testresults[0].T[0:6].T, tf)
plot_control_forces_vs_time(testresults[1], tf)

test_X, test_Q = forward_pass_dynamics(testresults[1])

plot_att_vs_time(test_X.T[6:13].T, tf)
plot_pos_vel_vs_time(test_X.T[0:6].T, tf)

### Measuring Inner Opt Runtime

In [ ]:
print(measure_opt_runtime(num_runs=5, subset_size=20))
print(measure_opt_runtime(num_runs=5, subset_size=24))
print(measure_opt_runtime(num_runs=5, subset_size=28))
print(measure_opt_runtime(num_runs=5, subset_size=32))

### Population Based Test

In [ ]:
ftauga = genetic_algorithm_optimization(pop_size=5, num_gens=1, warm_init=False)

In [ ]:
# Set-up hyperparameters
options = {'c1': 0.1, 'c2': 0.1, 'w': 0.07}

# Initialize the optimizer without bounds
optimizer = ps.single.GlobalBestPSO(n_particles=5, dimensions=3 * N, options=options)

# Set the initial positions of particles close to zero within [-1e-9, 1e-9]
optimizer.swarm.position = np.random.uniform(-1e-9, 1e-9, (5, 3 * N))

# Optimize
pso_cost, pso_tau_flat = optimizer.optimize(pso_function, iters=2)

# Reshape the result for `tau`
pso_tau = pso_tau_flat.reshape(N, 3)

print("Optimal tau:", pso_tau)
print("Optimal cost:", pso_cost)

In [ ]:
pso_tau = tau_proj_nonlin(pso_tau)[0]
psx,psu,psq,psc = opt_given_tau_ipopt(pso_tau)
pso_X, pso_Q = forward_pass_dynamics(psu)

violation = 0
print(pso_X[-1])
for i in range(0, 6):
    violation += np.linalg.norm(pso_X[-1][i] - xf[i])
for i in range(6, 10):  # Quaternion indices (q1 to q4)
    violation += min(np.linalg.norm(pso_X[-1][i] - xf[i]), np.linalg.norm(pso_X[-1][i] + xf[i]))
for i in range(10, 13):  # Angular velocity indices (omegas)
    violation += np.linalg.norm(pso_X[-1][i] - xf[i])

print(violation)

print(np.sum(np.square(psu)))

plot_control_forces_vs_time(psu, tf)
plot_pos_vel_vs_time(pso_X.T[0:6].T, tf)
plot_att_vs_time(pso_X.T[6:13].T, tf)

In [ ]:
ftauga = tau_proj_nonlin(ftauga)[0]
fxga, fuga, fqga, fcostga = opt_given_tau_ipopt(ftauga)
ga_X, ga_Q = forward_pass_dynamics(fuga)

violation = 0
print(fxga[-1])
for i in range(0, 6):
    violation += np.linalg.norm(ga_X[-1][i] - xf[i])
for i in range(6, 10):  # Quaternion indices (q1 to q4)
    violation += min(np.linalg.norm(ga_X[-1][i] - xf[i]), np.linalg.norm(ga_X[-1][i] + xf[i]))
for i in range(10, 13):  # Angular velocity indices (omegas)
    violation += np.linalg.norm(ga_X[-1][i] - xf[i])

print(violation)

print(np.sum(np.square(fuga)))

plot_control_forces_vs_time(fuga, tf)
plot_pos_vel_vs_time(ga_X.T[0:6].T, tf)
plot_att_vs_time(ga_X.T[6:13].T, tf)

### GA Test Code

In [ ]:
# lenient looper

def looper(pop_size=5, num_gens=5): #* time limit argument removed
    ftauga, time_taken, num_gens = genetic_algorithm_optimization(pop_size=5,num_gens=num_gens, warm_init=False) #*time limit argument removed
    ftauga = tau_proj_nonlin(ftauga)[0]
    fxga, fuga, fqga, fcostga = opt_given_tau_ipopt(ftauga)
    ga_X, ga_Q = forward_pass_dynamics(fuga)

    violation = 0
    print(fxga[-1])
    for i in range(0, 6):
        violation += np.linalg.norm(ga_X[-1][i] - xf[i])
    for i in range(6, 10):  # Quaternion indices (q1 to q4)
        violation += min(np.linalg.norm(ga_X[-1][i] - xf[i]), np.linalg.norm(ga_X[-1][i] + xf[i]))
    for i in range(10, 13):  # Angular velocity indices (omegas)
        violation += np.linalg.norm(ga_X[-1][i] - xf[i])

    print(violation)

    print(np.sum(np.square(fuga)))

    return violation, np.sum(np.square(fuga)), time_taken, num_gens

time_data=[]
violation_data=[]
cost_data=[]
gens_Data=[]


time_runs = [120]
gen_runs=[20, 30, 40]
iterations = 3
for limit in gen_runs:
    for i in range(iterations):
        print(f"iteration {i} for limit {limit}")
        v, c, t, g = looper(num_gens=limit)
        time_data.append(t)
        violation_data.append(v)
        cost_data.append(c)
        gens_Data.append(g)

print("Runs done!")
print("time data:")
print(time_data)
print("Violation Data")
print(violation_data)
print("Cost Data")
print(cost_data)
print("Gens Data")
print(gens_Data)

df = pd.DataFrame({"violation_data" : violation_data, "cost_data" : cost_data, "time_data" : time_data, "Gens_Data": gens_Data})
df.to_csv("extended_test_run_cga_1.csv", index=False)

### NLP Test

In [ ]:
funlp, fxnlp, fqnlp = full_nlp(time_limit=60, pointing=True)
print(funlp)

In [ ]:
starter_tau = tau_proj_nonlin(np.zeros((N,3)))[0] #projecting zeros
starter_x, starter_u, starter_q, starter_cost = opt_given_tau_ipopt(starter_tau) #optimising for projected tau

In [ ]:
funlp, fxnlp, fqnlp = full_nlp(time_limit = 30, U_guess=funlp, X_guess=np.concatenate([starter_x.flatten(),starter_q.flatten()]), pointing=True)

In [ ]:
# NLP Constraint Violation Checker

nlp_X, nlp_Q = forward_pass_dynamics(funlp)

violation = 0
print(nlp_X[-1])
for i in range(0, 6):
    violation += np.linalg.norm(nlp_X[-1][i] - xf[i])
for i in range(6, 10):  # Quaternion indices (q1 to q4)
    violation += min(np.linalg.norm(nlp_X[-1][i] - xf[i]), np.linalg.norm(nlp_X[-1][i] + xf[i]))
for i in range(10, 13):  # Angular velocity indices (omegas)
    violation += np.linalg.norm(nlp_X[-1][i] - xf[i])

print(f"violation for NLP is: {violation}")

print(np.sum(np.square(funlp)))


In [ ]:
# NLP Constraint Violation Checker

violation_sol = 0
print(fxnlp[-1])
for i in range(0, 6):
    violation_sol += np.linalg.norm(fxnlp[-1][i] - xf[i])
for i in range(6, 10):  # Quaternion indices (q1 to q4)
    violation_sol += min(np.linalg.norm(fxnlp[-1][i] - xf[i]), np.linalg.norm(fxnlp[-1][i] + xf[i]))
for i in range(10, 13):  # Angular velocity indices (omegas)
    violation_sol += np.linalg.norm(fxnlp[-1][i] - xf[i])

print(f"violation for NLP is: {violation_sol}")

print(np.sum(np.square(funlp)))



In [ ]:
# Cold Start Long RUnner code

import time

violation_fpd=[]
violation_sol=[]
cost_Data=[]
time_Data=[]

time_runs=[60]
iterations=5

for limit in time_runs:
    for i in range(iterations):
        print(f"Limit {limit} iteration {i}")
        start_time = time.time()
        funlp, fxnlp, fqnlp = full_nlp(time_limit=limit, pointing=True)
        end_time=time.time()

        #now calculating violation from solver
        viol_sol=0
        viol_fpd=0
        for i in range(0, 6):
            viol_sol += np.linalg.norm(fxnlp[-1][i] - xf[i])
        for i in range(6, 10):  # Quaternion indices (q1 to q4)
            viol_sol += min(np.linalg.norm(fxnlp[-1][i] - xf[i]), np.linalg.norm(fxnlp[-1][i] + xf[i]))
        for i in range(10, 13):  # Angular velocity indices (omegas)
            viol_sol += np.linalg.norm(fxnlp[-1][i] - xf[i])
        violation_sol.append(viol_sol)

        #now calculating violation from FPD

        nlp_X, nlp_Q = forward_pass_dynamics(funlp)
        for i in range(0, 6):
            viol_fpd += np.linalg.norm(nlp_X[-1][i] - xf[i])
        for i in range(6, 10):  # Quaternion indices (q1 to q4)
            viol_fpd += min(np.linalg.norm(nlp_X[-1][i] - xf[i]), np.linalg.norm(nlp_X[-1][i] + xf[i]))
        for i in range(10, 13):  # Angular velocity indices (omegas) 
            viol_fpd += np.linalg.norm(nlp_X[-1][i] - xf[i])
        violation_fpd.append(viol_fpd)

        cost_Data.append(np.sum(np.square(funlp)))
        time_Data.append(end_time-start_time)

print("Runs Done!")
print(f"Violation from Solver: {violation_sol}")
print(f"violation form fpd: {violation_fpd}")

df = pd.DataFrame({"violation_data_solver" : violation_sol, "violation_data_fpd" : violation_fpd, "cost_data" : cost_Data, "time_data" : time_Data})
df.to_csv("test_cold_nlp_oldcode_s3_60x5.csv", index=False)

In [ ]:
# Warm Start Long RUnner code

import time

violation_fpd=[]
violation_sol=[]
cost_Data=[]
time_Data=[]
returndata=[]
success=[]

time_runs=[600]
iterations=5

for limit in time_runs:
    for i in range(iterations):
        print(f"Limit {limit} iteration {i}")
        starter_tau = tau_proj_nonlin(np.zeros((N,3)))[0] #projecting zeros
        starter_x, starter_u, starter_q, starter_cost = opt_given_tau_ipopt(starter_tau) #optimising for projected tau
        start_time = time.time()
        funlp, fxnlp, fqnlp, ret, suc = full_nlp(time_limit=limit,  U_guess=starter_u, X_guess=np.concatenate([starter_x.flatten(),starter_q.flatten()]), pointing=True)
        end_time=time.time()
        returndata.append(ret)
        success.append(suc)
        #now calculating violation from solver
        viol_sol=0
        viol_fpd=0
        for i in range(0, 6):
            viol_sol += np.linalg.norm(fxnlp[-1][i] - xf[i])
        for i in range(6, 10):  # Quaternion indices (q1 to q4)
            viol_sol += min(np.linalg.norm(fxnlp[-1][i] - xf[i]), np.linalg.norm(fxnlp[-1][i] + xf[i]))
        for i in range(10, 13):  # Angular velocity indices (omegas)
            viol_sol += np.linalg.norm(fxnlp[-1][i] - xf[i])
        violation_sol.append(viol_sol)

        #now calculating violation from FPD

        nlp_X, nlp_Q = forward_pass_dynamics(funlp)
        for i in range(0, 6):
            viol_fpd += np.linalg.norm(nlp_X[-1][i] - xf[i])
        for i in range(6, 10):  # Quaternion indices (q1 to q4)
            viol_fpd += min(np.linalg.norm(nlp_X[-1][i] - xf[i]), np.linalg.norm(nlp_X[-1][i] + xf[i]))
        for i in range(10, 13):  # Angular velocity indices (omegas) 
            viol_fpd += np.linalg.norm(nlp_X[-1][i] - xf[i])
        violation_fpd.append(viol_fpd)

        cost_Data.append(np.sum(np.square(funlp)))
        time_Data.append(end_time-start_time)

print("Runs Done!")
print(f"Violation from Solver: {violation_sol}")
print(f"violation form fpd: {violation_fpd}")
print(f"Statuses: {returndata}")

df = pd.DataFrame({"violation_data_solver" : violation_sol, "violation_data_fpd" : violation_fpd, "cost_data" : cost_Data, "time_data" : time_Data, "return data" : returndata, "successdata" : success})
df.to_csv("test_warm_nlp_oldcode_s2_600x5_2.csv", index=False)